# Grayscale Gaussian denoising with an RGB diffusion prior: the isotropic chroma-noise lift (pre-registered kill test)

A gray observation y = x + σn is lifted to RGB as Y = y·1 + w with w ~ N(0, 3σ²(I − uuᵀ)), u = (1,1,1)/√3. The
noise in Y is then exactly isotropic with per-channel std √3σ, and the luminance (the sufficient statistic) is
unchanged. The one-step ADM posterior mean at the step matched to √3σ is averaged over channels; K = 2 chroma draws
× horizontal flip (4 evaluations) is the pre-registered estimator, with its nested 1- and 2-evaluation subsets
reported as ablations. A DMID-style arm (y replicated to 3 channels, no chroma noise, step at σ) is the naive lift.

Protocol (Restormer's `test_gaussian_gray_denoising.py`): float32 image / 255, `np.random.seed(seed)` per image,
`img += np.random.normal(0, σ/255, shape)`, no clipping; outputs clamped and rounded to uint8. Seed 0 is the
published protocol; seeds 1 and 2 replicate. Restormer-gray (σ-specific) is rerun on the same draws as the validity
check: it must land within ±0.03 dB of its published numbers at seed 0.

Pre-registered (before any score): success = BSD68 ≥ 29.60 / 26.70 dB at σ 25 / 50 (published best + 0.05) and
≥ +0.08 paired over Restormer-gray; kill = below published best + 0.02 at both σ, or below +0.03 over Restormer-gray.
Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import importlib.util, json, math, os, queue, subprocess, sys, threading, time, traceback, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

SIGMAS = [15, 25, 50]
SEEDS = [0, 1, 2]
K_DRAWS = 2                         # chroma-noise draws; each with its horizontal flip
TIME_LIMIT_H = 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)


def download(url, path, size=None):
    """Download with retries; no-op when the file is already complete."""
    if os.path.exists(path) and (size is None or os.path.getsize(path) == size) and os.path.getsize(path) > 0:
        return path
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(path, "wb") as f:
                while chunk := r.read(1 << 23):
                    f.write(chunk)
            if size is None or os.path.getsize(path) == size:
                return path
        except Exception as e:
            print(f"  retry {attempt + 1} for {url}: {e}", flush=True)
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"download failed: {url}")


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


t0 = time.time()
GH = "https://github.com/{}/releases/download/{}/{}"
FILES = {"adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297)}
for s in SIGMAS:
    FILES[f"restormer_gray_{s}"] = (GH.format("swz30/Restormer", "v1.0", f"gaussian_gray_denoising_sigma{s}.pth"), None)
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
IMAGES = {"Set12": [f"{i:02d}.png" for i in range(1, 13)], "BSD68": [f"test{i:03d}.png" for i in range(1, 69)]}
RESTORMER_SRC = f"{CACHE}/restormer_arch.py"
download("https://raw.githubusercontent.com/swz30/Restormer/68dc6ac472db26f16361150cb7a96a1bc87da93f/basicsr/models/archs/restormer_arch.py",
         RESTORMER_SRC)
with ThreadPoolExecutor(8) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
    paths = {(ds, n): ex.submit(download, RAW.format(ds, n), f"{CACHE}/{ds}_{n}") for ds, ns in IMAGES.items() for n in ns}
    paths = {k: v.result() for k, v in paths.items()}
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TEST = {ds: [(n, np.asarray(Image.open(paths[(ds, n)]).convert("L"))) for n in ns] for ds, ns in IMAGES.items()}
TIMES["downloads"] = time.time() - t0
print({ds: len(v) for ds, v in TEST.items()}, "sizes:", sorted({im.shape for v in TEST.values() for _, im in v}),
      f"downloads {TIMES['downloads'] / 60:.1f} min")

## 1 · Noise (Restormer / KAIR gray protocol) and metrics

In [ ]:
def noisy_of(clean_u8, sigma, seed):
    """float32 image / 255 plus np.random.normal(0, σ/255) after np.random.seed(seed), added in place (float32)."""
    img = np.float32(clean_u8[..., None]) / 255.
    np.random.seed(seed=seed)
    img += np.random.normal(0, sigma / 255., img.shape)
    return torch.from_numpy(img[..., 0].copy())


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None]
    conv = lambda x: F.conv2d(x[None, None], w)[0, 0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def to_u8(x):
    """[H,W] float in [0,1] units -> clamped, rounded float on the 0-255 scale."""
    return (x.clamp(0, 1) * 255).round()


def scores(out_u8, ref_u8):
    a, b = out_u8.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf"), ssim=ssim(a, b))

## 2 · Networks on every GPU: ADM (fp16, SDPA on contiguous q, k, v) and Restormer-gray σ-specific (fp32)

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.transpose(1, 2).contiguous() for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)
RestormerNet = load_module("restormer_arch", RESTORMER_SRC).Restormer


def make_adm(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                     use_scale_shift_norm=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False).convert_to_fp16()
    return m


def make_restormers(device):
    nets = {}
    for s in SIGMAS:
        net = RestormerNet(inp_channels=1, out_channels=1, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4, heads=[1, 2, 4, 8],
                           ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)
        net.load_state_dict(torch.load(P[f"restormer_gray_{s}"], map_location="cpu", weights_only=True)["params"], strict=True)
        nets[s] = net.to(device).eval().requires_grad_(False)
    return nets


CTX = [dict(dev=d, adm=make_adm(d), restormer=make_restormers(d)) for d in DEVICES]
ADM_STATE.clear()

## 3 · Estimators

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()
U = torch.full((3,), 1 / math.sqrt(3))


def matched(sigma01):
    """Continuous step t and abar with abar = 1 / (1 + (2σ)^2)."""
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


@torch.no_grad()
def tweedie(model, y, sigma01):
    """One-step posterior mean of RGB y [B,3,H,W] ([0,1] units, unclipped) at the step matched to sigma01."""
    t, ab = matched(sigma01)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3].float()
    return (((x - math.sqrt(1 - ab) * eps) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


def chroma_noise(shape_hw, sigma01, gen, device):
    """w ~ N(0, 3σ²(I − uuᵀ)) per pixel: the u-component of a white RGB draw removed, scaled by √3σ."""
    z = torch.randn((3, *shape_hw), generator=gen)
    z = z - U[:, None, None] * (U[:, None, None] * z).sum(0, keepdim=True)
    return (math.sqrt(3) * sigma01 * z).to(device)


@torch.no_grad()
def restormer_gray(net, y):
    """Restormer's script: reflect-pad to a multiple of 8, clamp to [0, 1]."""
    h, w = y.shape[-2:]
    yp = F.pad(y[None, None], (0, (-w) % 8, 0, (-h) % 8), mode="reflect")
    return net(yp)[0, 0, :h, :w].clamp(0, 1)

## 4 · Every estimator on every image, σ and seed

In [ ]:
def run_image(ctx, job):
    ds, idx, sigma, seed = job
    name, clean = TEST[ds][idx]
    dev, adm = ctx["dev"], ctx["adm"]
    y = noisy_of(clean, sigma, seed).to(dev)
    ref = torch.from_numpy(clean).to(dev).float()
    s01 = sigma / 255
    gen = torch.Generator().manual_seed(10_000_000 * (seed + 1) + 100_000 * SIGMAS.index(sigma) + 1000 * list(IMAGES).index(ds) + idx)
    outs, errors = {}, []
    try:
        lifted = []
        for _ in range(K_DRAWS):
            Y = y[None, None].expand(1, 3, -1, -1) + chroma_noise(y.shape, s01, gen, dev)[None]
            pair = tweedie(adm, torch.cat([Y, Y.flip(-1)]), math.sqrt(3) * s01)
            lifted += [pair[0].mean(0), pair[1].flip(-1).mean(0)]
        outs["ADM lift ×1"] = lifted[0]
        outs["ADM lift ×2 (flip)"] = (lifted[0] + lifted[1]) / 2
        outs[f"ADM lift ×{2 * K_DRAWS} ({K_DRAWS} draws × flip)"] = torch.stack(lifted).mean(0)
    except Exception:
        errors.append(("lift", traceback.format_exc(limit=3)))
    try:
        outs["ADM replicate, step at σ (DMID-style)"] = tweedie(adm, y[None, None].expand(1, 3, -1, -1), s01)[0].mean(0)
    except Exception:
        errors.append(("replicate", traceback.format_exc(limit=3)))
    try:
        outs["Restormer-gray"] = restormer_gray(ctx["restormer"][sigma], y)
    except Exception:
        errors.append(("restormer", traceback.format_exc(limit=3)))
    rows = [dict(dataset=ds, image=name, sigma=sigma, seed=seed, method=m, **scores(to_u8(o), ref)) for m, o in outs.items()]
    return rows, errors


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, failures)."""
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    while any(th.is_alive() for th in threads):
        time.sleep(10)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
jobs = [(ds, i, s, seed) for seed in SEEDS for s in SIGMAS for ds in IMAGES for i in range(len(IMAGES[ds]))]
res, fail = worker_pool(jobs[:2], run_image)
print("smoke test:", len(res), "ok,", len(fail), "failed;", [e for _, errs in res for e in errs][:2])
res2, fail2 = worker_pool(jobs[2:], run_image)
res, fail = res + res2, fail + fail2
SCORES = pd.DataFrame([r for rows, _ in res for r in rows])
METHOD_ERRORS = [e for _, errs in res for e in errs]
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
TIMES["denoising"] = time.time() - t0
print(f"{len(res)}/{len(jobs)} jobs, {len(fail)} failed jobs, {len(METHOD_ERRORS)} failed methods, {TIMES['denoising'] / 60:.1f} min")
for e in METHOD_ERRORS[:3]:
    print("METHOD ERROR", e)

## 5 · Results against the pre-registered criteria

In [ ]:
PUBLISHED = {  # σ 15 / 25 / 50
    "BSD68": {"Restormer": (31.96, 29.52, 26.62), "best": (32.00, 29.55, 26.65)},     # best: GRL-B (σ 15), Xformer (σ 25, 50)
    "Set12": {"Restormer": (33.42, 31.08, 28.00), "best": (33.48, 31.19, 28.15)},     # best: Hi-IR
}
mean = SCORES.groupby(["dataset", "method", "seed", "sigma"]).psnr.mean().unstack()
print("mean PSNR per seed")
print(mean.round(3).to_string())
seed_mean = SCORES.groupby(["dataset", "method", "sigma"]).psnr.mean().unstack()
print("\nmean PSNR over seeds")
print(seed_mean.round(3).to_string())
main = f"ADM lift ×{2 * K_DRAWS} ({K_DRAWS} draws × flip)"
wide = SCORES.pivot_table(index=["dataset", "image", "sigma", "seed"], columns="method", values="psnr")
paired = (wide[main] - wide["Restormer-gray"]).groupby(level=["dataset", "sigma"]).agg(["mean", "std", "count", lambda g: int((g > 0).sum())])
paired.columns = ["mean", "sd", "n", "better"]
print("\npaired", main, "− Restormer-gray (all seeds)")
print(paired.round(4).to_string())
verdict = {}
for ds, pub in PUBLISHED.items():
    for k, s in enumerate(SIGMAS):
        rerun0 = float(mean.loc[(ds, "Restormer-gray", 0), s])
        ours = float(seed_mean.loc[(ds, main), s])
        verdict[f"{ds} σ{s}"] = dict(ours=ours, best=pub["best"][k], margin=ours - pub["best"][k], restormer_rerun_seed0=rerun0,
                                     restormer_published=pub["Restormer"][k], rerun_gap=rerun0 - pub["Restormer"][k],
                                     paired_vs_restormer=float(paired.loc[(ds, s), "mean"]))
VERDICT = pd.DataFrame(verdict).T
print("\npre-registered check (BSD68 σ 25 / 50 decide)")
print(VERDICT.round(3).to_string())
b25, b50 = VERDICT.loc["BSD68 σ25"], VERDICT.loc["BSD68 σ50"]
valid = all(abs(VERDICT.loc[f"BSD68 σ{s}", "rerun_gap"]) <= 0.03 for s in (25, 50))
success = b25.ours >= 29.60 and b50.ours >= 26.70 and b25.paired_vs_restormer >= 0.08 and b50.paired_vs_restormer >= 0.08
kill = (b25.margin < 0.02 and b50.margin < 0.02) or (b25.paired_vs_restormer < 0.03 and b50.paired_vs_restormer < 0.03)
print(f"\nvalid (Restormer rerun within 0.03 at seed 0): {valid}; success: {success}; kill: {kill}")
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, verdict=VERDICT.reset_index().to_dict("records"), valid=bool(valid),
                   success=bool(success), kill=bool(kill), failed_jobs=[j for j, _ in fail], failed_methods=[e[0] for e in METHOD_ERRORS]),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))